In [1]:
# 清理显存占用
import torch, gc
if 'model' in globals():
    del model
if 'data' in globals():
    # 如果 data 很大，也可以考虑 del，但通常 data 本身只占 1-2GB
    pass

gc.collect()
torch.cuda.empty_cache()

In [3]:
import os
import os.path as osp
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv, MessagePassing
from torch_geometric.utils import negative_sampling
import torch_geometric.transforms as T
from ogb.nodeproppred import PygNodePropPredDataset, Evaluator # 引入 OGB 库
import numpy as np


# ==========================================
# global参数配置
# ==========================================
class Config:
    # 1. 基础参数
    hidden_dim = 32      # 节点嵌入维度
    proj_dim = 256       # 采样器投影维度
    hops = 2             # 递归深度
    
    # 2. 训练
    epochs = 500         # 总轮数
    soft_end =200       # Soft模式结束轮数 
    lr_decay_epoch = 300 # 学习率减半轮数
    lr_init = 0.001      # 初始学习率
    
    # 3. 损失函数
    target_sparsity = 0.50 
    lambda_max = 3.0       # sparsity 损失的最大系数
    link_weight = 1.0      
    weight_decay = 5e-4    # 权重衰减（1e-3）

# ==========================================
# 模块一至三 (SubgraphEnhancer, BilinearSampler, NeuralRecursiveSystem)
# ==========================================
class SubgraphEnhancer(MessagePassing):
    def __init__(self, in_channels, out_channels):
        super().__init__(aggr='add')
        self.lin = torch.nn.Linear(in_channels, out_channels)
    def forward(self, x, edge_index, weights):
        x = self.lin(x)
        return self.propagate(edge_index, x=x, weights=weights)
    def message(self, x_j, weights):
        return weights.view(-1, 1) * x_j

class BilinearSampler(nn.Module):
    def __init__(self, in_dim, proj_dim=128):  # 线性投影层可以增加
        super().__init__()
        self.proj = nn.Linear(in_dim, proj_dim)
        self.W = nn.Parameter(torch.Tensor(proj_dim, proj_dim))
        nn.init.xavier_uniform_(self.W)
    def forward(self, h_i, h_j):
        z_i = self.proj(h_i)
        z_j = self.proj(h_j)
        score = torch.sum((z_i @ self.W) * z_j, dim=-1) 
        return torch.sigmoid(score)

class NeuralRecursiveSystem(torch.nn.Module):
    def __init__(self, in_size, hidden_size, out_size, hops=2, tau=0.8): 
        super().__init__()
        # --- 双层 GAT 架构 ---
        # 第一层：将输入映射到 hidden_size * 8
        self.gat1 = GATConv(in_size, hidden_size, heads=8, dropout=0.6)
        self.ln1 = nn.LayerNorm(hidden_size * 8)
        
        # 第二层：在 hidden_size * 8 空间内进一步提取高阶特征
        # in 和 out 都是 hidden_size x 8，为了保持维度一致方便做残差
        self.gat2 = GATConv(hidden_size * 8, hidden_size, heads=8, dropout=0.6)
        self.ln2 = nn.LayerNorm(hidden_size * 8)
        
        # 采样器输入维度：GAT 2 的输出维度 (hidden_size x 8)
        self.sampler_net = BilinearSampler(in_dim=hidden_size * 8, proj_dim=Config.proj_dim)
        
        # 子图增强器
        self.enhancer = SubgraphEnhancer(hidden_size * 8, hidden_size * 8)
        
        # 残差连接的投影层：将原始输入 x 映射到与 GAT 输出一致的维度
        self.res_lin = nn.Linear(in_size, hidden_size * 8)
        
        self.link_predictor = torch.nn.Sequential(
            torch.nn.Linear((hidden_size * 8) * 2, 64),
            torch.nn.ReLU(),
            torch.nn.Linear(64, 1),
            torch.nn.Sigmoid()
        )
        
        self.classifier = torch.nn.Sequential(
            torch.nn.Linear(hidden_size * 8, hidden_size * 4),
            torch.nn.BatchNorm1d(hidden_size * 4),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.5),
            torch.nn.Linear(hidden_size * 4, out_size)
        )
        
        self.hops = hops
        self.tau = tau

    def get_neural_recursive_weights(self, h, edge_index, start_mask, hard=True):
        row, col = edge_index
        num_nodes = h.size(0)
        num_edges = edge_index.size(1)
        
        # 初始分数：基于双层GAT后的语义特征
        scores = self.sampler_net(h[row], h[col])
        
        final_weights = torch.zeros(num_edges, device=edge_index.device)
        # 修复：采样起点应该是当前需要预测的所有目标节点
        active_nodes = start_mask.float() 
        
        for h_step in range(self.hops):
            # 只有源节点被激活的边才参与计算
            active_edges_mask = active_nodes[row]
            
            # Gumbel-Softmax 采样
            # 增加 epsilon 防止 log(0)
            sampling_logits = torch.stack([1 - scores, scores], dim=-1).clamp(min=1e-9).log()
            sampling_mask = F.gumbel_softmax(sampling_logits, tau=self.tau, hard=hard, dim=-1)[:, 1]
            
            # 当前跳步激活的边权重
            current_step_weights = sampling_mask * active_edges_mask
            final_weights = torch.max(final_weights, current_step_weights)
            
            # 逻辑：统计哪些目标节点 (col) 被当前激活的边指向了
            new_active_nodes = torch.zeros(num_nodes, device=edge_index.device)
            new_active_nodes.scatter_add_(0, col, current_step_weights)
            
            # 只要节点被选中的边指向，它就成为下一轮递归的起点
            active_nodes = (new_active_nodes > 1e-5).float() 
            # ==========================================
            
        return final_weights

    def forward(self, x, edge_index, target_mask, hard=True):
        # 1. 基础特征提取 (语义发现)
        h1 = F.elu(self.ln1(self.gat1(x, edge_index)))
        h_base = self.gat2(h1, edge_index)
        h_base = F.elu(self.ln2(h_base) + self.res_lin(x)) # 残差
        
        # 2. 递归采样过程
        # 注意：target_mask 在训练时是 train_mask，测试时应是全图或 test_mask
        weights = self.get_neural_recursive_weights(h_base, edge_index, target_mask, hard=hard)
        
        # 3. 语义增强过程 (增加 LayerNorm 稳定深层递归)
        h_enhanced = h_base
        for _ in range(self.hops):
            msg = self.enhancer(h_enhanced, edge_index, weights)
            h_enhanced = h_enhanced + F.elu(msg) 
            # 建议在这里加入 LayerNorm(h_enhanced) 如果 hidden_dim 较大
            
        # 4. 输出
        logits = self.classifier(h_enhanced)
        
        # 链路预测：使用增强后的特征
        row, col = edge_index
        pos_edge_feat = torch.cat([h_enhanced[row], h_enhanced[col]], dim=-1)
        link_probs_pos = self.link_predictor(pos_edge_feat).squeeze()
        
        return F.log_softmax(logits, dim=1), link_probs_pos, h_enhanced, weights


# ==========================================
# 模块四：数据加载 (ogbn-arxiv)
# ==========================================
dataset_name = 'ogbn-arxiv'
# Arxiv 有向图转为无向图以增强消息传递能力
dataset = PygNodePropPredDataset(name=dataset_name, root='data', transform=T.ToUndirected())
data = dataset[0]
split_idx = dataset.get_idx_split()
train_idx = split_idx['train']
hidden_dim = 32
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = NeuralRecursiveSystem(dataset.num_features, hidden_size=hidden_dim, out_size=dataset.num_classes, hops=2).to(device)

data = data.to(device)
evaluator = Evaluator(name='ogbn-arxiv')

# 适配 train_mask
data.train_mask = torch.zeros(data.num_nodes, dtype=torch.bool, device=device)
data.train_mask[train_idx] = True

optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)


# ==========================================
# 模块五：训练 
# ==========================================
def train(epoch, total_epochs, start_tau, end_tau):
    model.train()
    
    # 在函数内部定义训练所需的 mask，或者从外部传入
    # 为了保证训练效率，训练时通常使用 data.train_mask
    current_mask = data.train_mask 

    # --- 学习率调整逻辑 ---
    if epoch == Config.lr_decay_epoch:
        for param_group in optimizer.param_groups:
            param_group['lr'] = Config.lr_init * 0.5
            print(f"\n>>> Epoch {epoch}: 触发微调，学习率调整为 {param_group['lr']}")

    # --- 模式切换逻辑 ---
    use_hard = True if epoch > Config.soft_end else False
    curr_tau = max(end_tau, start_tau - (epoch / total_epochs) * (start_tau - end_tau))
    model.tau = curr_tau

    # --- Lambda 调度 ---
    if epoch <= (Config.soft_end / 3):
        curr_lambda = 0.1
    elif epoch <= Config.soft_end:
        progress = (epoch - Config.soft_end / 3) / (Config.soft_end * 2 / 3)
        curr_lambda = 0.1 + progress * (Config.lambda_max - 0.1)
    else:
        curr_lambda = Config.lambda_max

    optimizer.zero_grad()
    
    # 传入定义的 current_mask
    log_probs, link_probs_pos, h_enhanced, weights = model(data.x, data.edge_index, current_mask, hard=use_hard)
    
    # 负采样与链路预测
    neg_edge_index = negative_sampling(data.edge_index, num_nodes=data.num_nodes).to(device)
    neg_row, neg_col = neg_edge_index
    neg_edge_feat = torch.cat([h_enhanced[neg_row], h_enhanced[neg_col]], dim=-1)
    link_probs_neg = model.link_predictor(neg_edge_feat).squeeze()
    
    # Loss 计算
    loss_clf = F.nll_loss(log_probs[train_idx], data.y.squeeze()[train_idx])
    loss_link = F.binary_cross_entropy(link_probs_pos, torch.ones_like(link_probs_pos)) + \
                F.binary_cross_entropy(link_probs_neg, torch.zeros_like(link_probs_neg))
    
    current_sp = weights.mean()
    loss_sparse = F.mse_loss(current_sp, torch.tensor(Config.target_sparsity).to(weights.device))
    
    total_loss = loss_clf + Config.link_weight * loss_link + curr_lambda * loss_sparse
    
    # 防坍塌
    if current_sp < 0.05:
        total_loss += (0.1 - current_sp) * 20 

    total_loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    
    actual_sparsity = (weights > 0.5).float().mean().item()
    return total_loss.item(), loss_clf.item(), loss_link.item(), loss_sparse.item(), actual_sparsity, curr_lambda

# ==========================================
# 运行实验 (保存最佳模型)
# ==========================================

best_valid_acc = 0.0
best_model_path = 'best_model.pt'

history = {
    'train_loss': [], 'clf_loss': [], 'link_loss': [], 
    'sp_loss': [], 'sparsity': [], 'lambda': [],
    'train_acc': [], 'valid_acc': [], 'test_acc': []
}

print(f"开始 {dataset_name} 训练（目标稀疏度: {Config.target_sparsity}）...")


for epoch in range(1, Config.epochs + 1):
    t_loss, c_loss, l_loss, s_loss, sp_rate, c_lam = train(epoch, Config.epochs, 1.0, 0.1)
    
    history['train_loss'].append(t_loss)
    history['clf_loss'].append(c_loss)
    history['link_loss'].append(l_loss)
    history['sp_loss'].append(s_loss)
    history['sparsity'].append(sp_rate)
    history['lambda'].append(c_lam)
    
    if epoch % 10 == 0:
        model.eval()
        with torch.no_grad():
            full_mask = torch.ones(data.num_nodes, dtype=torch.bool, device=device)
            lp, _, _, _ = model(data.x, data.edge_index, full_mask, hard=True)
            y_pred = lp.argmax(dim=-1, keepdim=True)
            
            train_acc = evaluator.eval({'y_true': data.y[split_idx['train']], 'y_pred': y_pred[split_idx['train']]})['acc']
            valid_acc = evaluator.eval({'y_true': data.y[split_idx['valid']], 'y_pred': y_pred[split_idx['valid']]})['acc']
            test_acc = evaluator.eval({'y_true': data.y[split_idx['test']], 'y_pred': y_pred[split_idx['test']]})['acc']
            
            history['train_acc'].append(train_acc)
            history['valid_acc'].append(valid_acc)
            history['test_acc'].append(test_acc)
            
            # --- 保存最佳模型逻辑 ---
            if valid_acc > best_valid_acc:
                best_valid_acc = valid_acc
                torch.save(model.state_dict(), best_model_path)
                print(f"--- 发现更优模型: Epoch {epoch}, Valid Acc: {valid_acc:.4f}, 模型已保存 ---")

            mode_str = 'Hard' if epoch > Config.soft_end else 'Soft'
            curr_lr = optimizer.param_groups[0]['lr']
            print(f"Ep: {epoch:03d} | LR: {curr_lr:.5f} | λ: {c_lam:.2f} | Mode: {mode_str} | Sparsity: {sp_rate:.4f} | Train_acc: {train_acc:.4f} | Valid_acc: {valid_acc:.4f} | Test_acc: {test_acc:.4f}")

print(f"\n训练结束！最佳验证集准确率: {best_valid_acc:.4f}, 模型保存在: {best_model_path}")

/home/malina/.venv/lib/python3.8/site-packages/ogb/nodeproppred/dataset_pyg.py:69: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.data, self.slices = torch.load(self.pro

开始 ogbn-arxiv 训练（目标稀疏度: 0.5）...
--- 发现更优模型: Epoch 10, Valid Acc: 0.2920, 模型已保存 ---
Ep: 010 | LR: 0.00100 | λ: 0.10 | Mode: Soft | Sparsity: 0.0000 | Train_acc: 0.2831 | Valid_acc: 0.2920 | Test_acc: 0.2620
--- 发现更优模型: Epoch 20, Valid Acc: 0.3371, 模型已保存 ---
Ep: 020 | LR: 0.00100 | λ: 0.10 | Mode: Soft | Sparsity: 0.0277 | Train_acc: 0.3371 | Valid_acc: 0.3371 | Test_acc: 0.3071
--- 发现更优模型: Epoch 30, Valid Acc: 0.3418, 模型已保存 ---
Ep: 030 | LR: 0.00100 | λ: 0.10 | Mode: Soft | Sparsity: 0.0274 | Train_acc: 0.3510 | Valid_acc: 0.3418 | Test_acc: 0.3092
--- 发现更优模型: Epoch 40, Valid Acc: 0.4149, 模型已保存 ---
Ep: 040 | LR: 0.00100 | λ: 0.10 | Mode: Soft | Sparsity: 0.0730 | Train_acc: 0.4355 | Valid_acc: 0.4149 | Test_acc: 0.3920
--- 发现更优模型: Epoch 50, Valid Acc: 0.5580, 模型已保存 ---
Ep: 050 | LR: 0.00100 | λ: 0.10 | Mode: Soft | Sparsity: 0.1058 | Train_acc: 0.5347 | Valid_acc: 0.5580 | Test_acc: 0.5589
--- 发现更优模型: Epoch 60, Valid Acc: 0.6074, 模型已保存 ---
Ep: 060 | LR: 0.00100 | λ: 0.10 | Mode: Soft | 

KeyboardInterrupt: 

In [ ]:
# ==========================================
# 训练结束后：加载最佳模型并进行最终评估
# ==========================================
print("\n" + "="*30)
print("开始最终评估 (Final Evaluation)...")

# 1. 重新实例化一个模型结构并加载参数
final_model = NeuralRecursiveSystem(
    dataset.num_features, 
    hidden_size=Config.hidden_dim, 
    out_size=dataset.num_classes, 
    hops=Config.hops
).to(device)

final_model.load_state_dict(torch.load(best_model_path))
final_model.eval()

# 2. 全图推理
with torch.no_grad():
    full_mask = torch.ones(data.num_nodes, dtype=torch.bool, device=device)
    # 使用 hard=True 保证结构确定性
    final_log_probs, _, _, _ = final_model(data.x, data.edge_index, full_mask, hard=True)
    final_pred = final_log_probs.argmax(dim=-1, keepdim=True)

    # 3. 计算三项指标
    final_train_acc = evaluator.eval({'y_true': data.y[split_idx['train']], 'y_pred': final_pred[split_idx['train']]})['acc']
    final_valid_acc = evaluator.eval({'y_true': data.y[split_idx['valid']], 'y_pred': final_pred[split_idx['valid']]})['acc']
    final_test_acc = evaluator.eval({'y_true': data.y[split_idx['test']], 'y_pred': final_pred[split_idx['test']]})['acc']

print(f">>> [最佳模型结果] <<<")
print(f"Final Train Acc: {final_train_acc:.4f}")
print(f"Final Valid Acc: {final_valid_acc:.4f}")
print(f"Final Test Acc:  {final_test_acc:.4f}  <-- 这是你论文表格里填的数据")
print("="*30)

In [ ]:
# ==========================================
# 模块六：可视化
# ==========================================
import matplotlib.pyplot as plt

def plot_experimental_results(history, config):
    # 动态获取 X 轴，防止长度不匹配报错
    epochs_range = range(1, len(history['train_loss']) + 1)
    
    # 修复：基于 acc 列表的实际长度生成 range
    # 假设每 10 轮记录一次，步长为 10
    num_acc_records = len(history['train_acc'])
    acc_range = [i * 10 for i in range(1, num_acc_records + 1)] 

    plt.figure(figsize=(18, 5))

    # 图 1：各项 Loss 趋势
    plt.subplot(1, 3, 1)
    plt.plot(epochs_range, history['clf_loss'], label='Clf (NLL)', alpha=0.8)
    plt.plot(epochs_range, history['link_loss'], label='Link (BCE)', alpha=0.8)
    plt.plot(epochs_range, history['sp_loss'], label='Sparse (MSE)', alpha=0.8)
    plt.axvline(x=config.soft_end, color='r', linestyle='--', label='Hard Mode Switch')
    plt.title('Loss Components Breakdown')
    plt.xlabel('Epochs')
    plt.ylabel('Loss Value')
    plt.legend()
    plt.grid(True, linestyle=':', alpha=0.6)

    # 图 2：准确率对比
    plt.subplot(1, 3, 2)
    # 获取最高测试准确率及其发生的 Epoch
    if history['test_acc']:
        max_test_acc = max(history['test_acc'])
        best_epoch = acc_range[np.argmax(history['test_acc'])]
        plt.plot(acc_range, history['train_acc'], marker='.', label='Train Acc')
        plt.plot(acc_range, history['valid_acc'], marker='.', label='Valid Acc')
        plt.plot(acc_range, history['test_acc'], marker='.', label='Test Acc', color='red')
        plt.scatter(best_epoch, max_test_acc, color='black', s=100, zorder=5, label='Best Point')
        plt.title(f'Acc (Best Test: {max_test_acc:.4f} @ Ep {best_epoch})')
    
    plt.axvline(x=config.soft_end, color='r', linestyle='--')
    plt.axvline(x=config.lr_decay_epoch, color='g', linestyle='--', label='LR Decay')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True, linestyle=':', alpha=0.6)

    # 图 3：稀疏度演变
    plt.subplot(1, 3, 3)
    plt.plot(epochs_range, history['sparsity'], color='purple', label='Actual Sparsity')
    plt.axhline(y=config.target_sparsity, color='gray', linestyle='-', label='Target')
    plt.axvline(x=config.soft_end, color='r', linestyle='--', label='Hard Mode')
    
    # 增加平滑线，方便看趋势（可选）
    if len(history['sparsity']) > 20:
        smooth_sp = np.convolve(history['sparsity'], np.ones(10)/10, mode='valid')
        plt.plot(epochs_range[9:], smooth_sp, color='cyan', alpha=0.5, label='Trend (MA10)')
        
    plt.title(f'Sparsity (Target: {config.target_sparsity})')
    plt.xlabel('Epochs')
    plt.ylabel('Sparsity Rate')
    plt.legend()
    plt.grid(True, linestyle=':', alpha=0.6)

    plt.tight_layout()
    plt.show()

# 调用绘图
plot_experimental_results(history, Config)

In [ ]:
@torch.no_grad()
def evaluate_density_and_link_impact():
    model.eval()
    device = next(model.parameters()).device
    
    # 1. 前向传播：获取增强后的结果
    # 这里会运行完整的双层 GAT + 语义增强链条
    log_probs, link_probs_pos, h_enhanced, weights = model(data.x, data.edge_index, data.train_mask, hard=True)
    
    # 2. 计算密度
    total_edges = data.edge_index.size(1)
    active_edges = (weights > 0.5).sum().item()
    density = active_edges / total_edges
    
    # 3. 构造链路预测测试集
    neg_edge_index = negative_sampling(data.edge_index, num_nodes=data.num_nodes).to(device)
    neg_row, neg_col = neg_edge_index
    row, col = data.edge_index
    
    # --- 方案 A：基于采样增强特征 (h_enhanced) ---
    pos_enhanced_feat = torch.cat([h_enhanced[row], h_enhanced[col]], dim=-1)
    neg_enhanced_feat = torch.cat([h_enhanced[neg_row], h_enhanced[neg_col]], dim=-1)
    
    lp_pos_s = model.link_predictor(pos_enhanced_feat).squeeze()
    lp_neg_s = model.link_predictor(neg_enhanced_feat).squeeze()
    
    y_true = torch.cat([
        torch.ones(lp_pos_s.size(0), device=device), 
        torch.zeros(lp_neg_s.size(0), device=device)
    ])
    y_pred_s = torch.cat([lp_pos_s, lp_neg_s])
    sampler_impact_acc = ((y_pred_s > 0.5).float() == y_true).float().mean().item()
    
    # --- 方案 B：基于原始全局特征 (h_base) ---
    # 【核心修正】：手动模拟双层 GAT 的前向过程，但不包含最后的增强步骤
    h1 = F.elu(model.ln1(model.gat1(data.x, data.edge_index)))
    h_base = model.ln2(model.gat2(h1, data.edge_index))
    h_base = F.elu(h_base + model.res_lin(data.x)) # 必须加上残差，才是完整的全局表征
    
    pos_base_feat = torch.cat([h_base[row], h_base[col]], dim=-1)
    neg_base_feat = torch.cat([h_base[neg_row], h_base[neg_col]], dim=-1)
    
    lp_pos_b = model.link_predictor(pos_base_feat).squeeze()
    lp_neg_b = model.link_predictor(neg_base_feat).squeeze()
    
    y_pred_b = torch.cat([lp_pos_b, lp_neg_b])
    global_base_acc = ((y_pred_b > 0.5).float() == y_true).float().mean().item()
    
    return density, active_edges, total_edges, sampler_impact_acc, global_base_acc


# ==========================================
# 在训练 100 epoch 后的调用逻辑
# ==========================================
density, active, total, s_acc, g_acc = evaluate_density_and_link_impact()


print(f"【obgn-arxiv】")
print("-" * 50)
print(f"1. 采样密度分析 (Structural Sparsity):")
print(f"   - 激活边数 / 总边数: {active} / {total}")
print(f"   - 密度 (Density): {density:.2%}")
print("-" * 50)
print(f"2. 链路预测对比 (Link Prediction Impact):")
print(f"   - 全局基准 (无采样增强) Acc: {g_acc:.4f}")
print(f"   - 采样器 (经过增强表征) Acc: {s_acc:.4f}")
print(f"   - 性能提升 (Gain): {((s_acc - g_acc)/g_acc):.2%}")
print("-" * 50)

【obgn-arxiv】
--------------------------------------------------
1. 采样密度分析 (Structural Sparsity):
   - 激活边数 / 总边数: 776597 / 2315598
   - 密度 (Density): 33.54%
--------------------------------------------------
2. 链路预测对比 (Link Prediction Impact):
   - 全局基准 (无采样增强) Acc: 0.9124
   - 采样器 (经过增强表征) Acc: 0.9646
   - 性能提升 (Gain): 5.71%
--------------------------------------------------


In [ ]:
# 训练前强制清理
import gc
gc.collect()
torch.cuda.empty_cache()